In [1]:
from pathlib import Path

DATA_DIR = Path("../data/raw/MachineLearningCSV/MachineLearningCVE")

print("Dataset directory:")
print(DATA_DIR.resolve())

print("\nCSV files:")
for file in DATA_DIR.glob("*.csv"):
    print(f"{file.name} - {file.stat().st_size / (1024**2):.2f} MB")

Dataset directory:
C:\SentinelXAI\data\raw\MachineLearningCSV\MachineLearningCVE

CSV files:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv - 73.55 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv - 73.34 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv - 55.62 MB
Monday-WorkingHours.pcap_ISCX.csv - 168.73 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv - 79.25 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv - 49.61 MB
Tuesday-WorkingHours.pcap_ISCX.csv - 128.82 MB
Wednesday-workingHours.pcap_ISCX.csv - 214.74 MB


In [2]:
import pandas as pd

print("Dataset Structure Audit")
print("=" * 60)

for file in sorted(DATA_DIR.glob("*.csv")):
    df_sample = pd.read_csv(file, nrows=1000)

    print(f"\nFile: {file.name}")
    print(f"Rows sampled: {len(df_sample)}")
    print(f"Number of columns: {len(df_sample.columns)}")
    print("First 10 columns:")
    print(df_sample.columns[:10].tolist())
    print("Last 5 columns:")
    print(df_sample.columns[-5:].tolist())

Dataset Structure Audit

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Rows sampled: 1000
Number of columns: 79
First 10 columns:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std']
Last 5 columns:
['Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', ' Label']

File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows sampled: 1000
Number of columns: 79
First 10 columns:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std']
Last 5 columns:
['Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', ' Label']

File: Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows sampled: 1000
Nu

In [3]:
print("Label Distribution in 1,000-Row Samples")
print("=" * 60)

for file in sorted(DATA_DIR.glob("*.csv")):
    df_sample = pd.read_csv(file, nrows=1000)

    print(f"\n{file.name}")
    print(df_sample[" Label"].value_counts())

Label Distribution in 1,000-Row Samples

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64

Wednesday-workingHours.pcap_ISCX.csv
 Label
BENIGN    1000
Name: count, dtype: int64


In [4]:
print("Label Distribution from Multiple Parts of Each File")
print("=" * 70)

for file in sorted(DATA_DIR.glob("*.csv")):
    # Read the file in chunks without loading the whole file
    label_counts = {}

    for chunk in pd.read_csv(file, usecols=[" Label"], chunksize=100000):
        counts = chunk[" Label"].value_counts()

        for label, count in counts.items():
            label_counts[label] = label_counts.get(label, 0) + count

    print(f"\n{file.name}")
    for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {label}: {count:,}")

Label Distribution from Multiple Parts of Each File

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  DDoS: 128,027
  BENIGN: 97,718

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  PortScan: 158,930
  BENIGN: 127,537

Friday-WorkingHours-Morning.pcap_ISCX.csv
  BENIGN: 189,067
  Bot: 1,966

Monday-WorkingHours.pcap_ISCX.csv
  BENIGN: 529,918

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  BENIGN: 288,566
  Infiltration: 36

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  BENIGN: 168,186
  Web Attack � Brute Force: 1,507
  Web Attack � XSS: 652
  Web Attack � Sql Injection: 21

Tuesday-WorkingHours.pcap_ISCX.csv
  BENIGN: 432,074
  FTP-Patator: 7,938
  SSH-Patator: 5,897

Wednesday-workingHours.pcap_ISCX.csv
  BENIGN: 440,031
  DoS Hulk: 231,073
  DoS GoldenEye: 10,293
  DoS slowloris: 5,796
  DoS Slowhttptest: 5,499
  Heartbleed: 11


In [5]:
from collections import Counter

overall_labels = Counter()

for file in sorted(DATA_DIR.glob("*.csv")):
    for chunk in pd.read_csv(
        file,
        usecols=[" Label"],
        chunksize=100000
    ):
        overall_labels.update(chunk[" Label"].value_counts().to_dict())

label_df = pd.DataFrame(
    overall_labels.items(),
    columns=["Label", "Count"]
).sort_values("Count", ascending=False)

label_df["Percentage"] = (
    label_df["Count"] / label_df["Count"].sum() * 100
)

label_df

,Label,Count,Percentage
1,BENIGN,2273097,80.300366
10,DoS Hulk,231073,8.162981
2,PortScan,158930,5.614427
0,DDoS,128027,4.522735
13,DoS GoldenEye,10293,0.363615
8,FTP-Patator,7938,0.280421
9,SSH-Patator,5897,0.208320
11,DoS slowloris,5796,0.204752
12,DoS Slowhttptest,5499,0.194260
3,Bot,1966,0.069452


In [6]:
print("Data Quality Audit")
print("=" * 70)

total_rows = 0
missing_values = 0
infinite_values = 0
duplicate_rows = 0

for file in sorted(DATA_DIR.glob("*.csv")):
    print(f"\nProcessing: {file.name}")

    file_rows = 0
    file_missing = 0
    file_infinite = 0

    for chunk in pd.read_csv(file, chunksize=100000):

        file_rows += len(chunk)

        # Missing values
        file_missing += chunk.isna().sum().sum()

        # Infinite values in numeric columns
        numeric_data = chunk.select_dtypes(include="number")
        file_infinite += numeric_data.isin([float("inf"), float("-inf")]).sum().sum()

    total_rows += file_rows
    missing_values += file_missing
    infinite_values += file_infinite

    print(f"Rows: {file_rows:,}")
    print(f"Missing values: {file_missing:,}")
    print(f"Infinite values: {file_infinite:,}")

print("\n" + "=" * 70)
print(f"TOTAL ROWS: {total_rows:,}")
print(f"TOTAL MISSING VALUES: {missing_values:,}")
print(f"TOTAL INFINITE VALUES: {infinite_values:,}")

Data Quality Audit

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Rows: 225,745
Missing values: 4
Infinite values: 64

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows: 286,467
Missing values: 15
Infinite values: 727

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows: 191,033
Missing values: 28
Infinite values: 216

Processing: Monday-WorkingHours.pcap_ISCX.csv
Rows: 529,918
Missing values: 64
Infinite values: 810

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Rows: 288,602
Missing values: 18
Infinite values: 396

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Rows: 170,366
Missing values: 20
Infinite values: 250

Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Rows: 445,909
Missing values: 201
Infinite values: 327

Processing: Wednesday-workingHours.pcap_ISCX.csv
Rows: 692,703
Missing values: 1,008
Infinite values: 1,586

TOTAL ROWS: 2,830,743
TOTAL MISSING VALUES: 1,358
TOTAL INFINITE VALUES: 4,

In [7]:
print("Feature Data-Type Audit")
print("=" * 70)

# Read a sample from each file
for file in sorted(DATA_DIR.glob("*.csv")):
    df_sample = pd.read_csv(file, nrows=5000)

    # Remove whitespace from column names for inspection
    df_sample.columns = df_sample.columns.str.strip()

    non_numeric = df_sample.select_dtypes(exclude="number").columns.tolist()

    print(f"\n{file.name}")
    print(f"Non-numeric columns: {non_numeric}")

Feature Data-Type Audit

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Non-numeric columns: ['Label']

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Non-numeric columns: ['Label']

Friday-WorkingHours-Morning.pcap_ISCX.csv
Non-numeric columns: ['Label']

Monday-WorkingHours.pcap_ISCX.csv
Non-numeric columns: ['Label']

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Non-numeric columns: ['Label']

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Non-numeric columns: ['Label']

Tuesday-WorkingHours.pcap_ISCX.csv
Non-numeric columns: ['Label']

Wednesday-workingHours.pcap_ISCX.csv
Non-numeric columns: ['Label']
